In [1]:
%load_ext autoreload
%autoreload 2

from anomalib.data import MVTecAD
from anomalib.models import Patchcore
from anomalib.engine import Engine

In [2]:
# Initialize model and data
datamodule = MVTecAD(
    root="datasets/mvtec_anomaly_detection",
    category="cable",
)
model = Patchcore(
    backbone="resnet18",  # Lighter than wide_resnet50_2
    layers=["layer2"],  # Use fewer layers
    coreset_sampling_ratio=0.01  # Reduce to 1% instead of 10%
)

Train the model

In [3]:
import os

CHECKPOINT_PATH = "saved_model/patchcore_cable.ckpt"

engine = Engine(max_epochs=1, enable_progress_bar=False, logger=False)

if not os.path.exists(CHECKPOINT_PATH):
    engine.fit(model=model, datamodule=datamodule)
    os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
    engine.trainer.save_checkpoint(CHECKPOINT_PATH)
    print(f"Training complete. Checkpoint saved to '{CHECKPOINT_PATH}'.")
else:
    print(f"Checkpoint found. Skipping training.")


Checkpoint found. Skipping training.


In [4]:
import tkinter as tk
from tkinter import filedialog
import ipywidgets as widgets
from IPython.display import display

extra_training_images = []

def _open_picker(b):
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    paths = filedialog.askopenfilenames(
        title="Select defect-free training images",
        filetypes=[
            ("Images", "*.png *.jpg *.jpeg *.bmp *.tiff"),
            ("All files", "*.*"),
        ],
    )
    root.destroy()
    for p in paths:
        if p not in extra_training_images:
            extra_training_images.append(p)
    _refresh()

def _clear(b):
    extra_training_images.clear()
    _refresh()

def _refresh():
    if extra_training_images:
        rows = "".join(f"<li style='font-size:12px'>{p}</li>" for p in extra_training_images)
        status.value = f"<b>{len(extra_training_images)} image(s) selected:</b><ul>{rows}</ul>"
    else:
        status.value = "<i style='color:gray;'>No images selected yet.</i>"

pick_btn = widgets.Button(description="Browse for images...", button_style="primary", icon="folder-open")
clear_btn = widgets.Button(description="Clear selection", button_style="danger")
status = widgets.HTML("<i style='color:gray;'>No images selected yet.</i>")

pick_btn.on_click(_open_picker)
clear_btn.on_click(_clear)

display(widgets.VBox([
    widgets.HTML("<b>Select extra good (defect-free) training images:</b>"),
    widgets.HBox([pick_btn, clear_btn]),
    status,
]))

In [5]:
import os, shutil

TRAIN_GOOD_DIR = os.path.join(
    "datasets", "mvtec_anomaly_detection", "cable", "train", "good"
)
USER_PREFIX = "__user__"

# Use images selected by the file picker above, or fall back to an empty list
_images = globals().get("extra_training_images", [])

if not _images:
    print("No extra images selected — skipping retrain.")
else:
    # Remove any previously copied user images so we start fresh each run
    for fname in os.listdir(TRAIN_GOOD_DIR):
        if fname.startswith(USER_PREFIX):
            os.remove(os.path.join(TRAIN_GOOD_DIR, fname))

    # Copy selected images into the training folder
    copied = []
    for src in _images:
        dest = os.path.join(TRAIN_GOOD_DIR, USER_PREFIX + os.path.basename(src))
        shutil.copy(src, dest)
        copied.append(dest)
    print(f"Copied {len(copied)} extra image(s) to {TRAIN_GOOD_DIR}")

    # Re-fit a fresh PatchCore instance (rebuilds the coreset from scratch)
    model = Patchcore(
        backbone="resnet18",
        layers=["layer2"],
        coreset_sampling_ratio=0.01,
    )
    engine = Engine(max_epochs=1, enable_progress_bar=False, logger=False)
    engine.fit(model=model, datamodule=datamodule)

    # Overwrite checkpoint so the predict cell picks up the new model
    os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
    engine.trainer.save_checkpoint(CHECKPOINT_PATH)
    print(f"Retrain complete. Checkpoint updated at '{CHECKPOINT_PATH}'.")
    print("Re-run the cells below (predict -> process -> UI) to refresh results.")

No extra images selected — skipping retrain.


In [6]:
# Get predictions from ckpt file
predictions = engine.predict(model=model, datamodule=datamodule, ckpt_path=CHECKPOINT_PATH)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
Restoring states from the checkpoint path at C:\Users\ierem_wyrt65d\Documents\Samuel\Study\VUB\Bachelor_3\BA3_bachelorproef\bachelor_thesis\saved_model\patchcore_cable.ckpt
c:\Users\ierem_wyrt65d\Documents\Samuel\Study\VUB\Bachelor_3\BA3_bachelorproef\bachelor_thesis\.venv\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:566: The dirpath has changed from 'C:\\Users\\ierem_wyrt65d\\Documents\\Samuel\\Study\\VUB\\Bachelor_3\\BA3_bachelorproef\\bachelor_thesis\\results\\Patchcore\\MVTecAD\\cable\\v8\\weights\\lightning' to 'C:\\Users\\ierem_wyrt65d\\Documents\\Samuel\\Study\\VUB\\Bachelor_3\\BA3_bachelorproef\\bachelor_thesis\\results\\Patchcore\\MVTecAD\\cable\\latest\\weights\\lightning', therefore `best_model_score`, `kth_best_model_path`, `kth_value`, `last_model_path` and `best_k_models` won't be reloaded. Only `best_model_path` will be reloaded.
Loaded model weights from the checkpoint at C:\

In [7]:
import os
import shutil
import numpy as np
from PIL import Image as PILImage
import help_functions as utils

# Create a clean folder structure
base_path = "output_results"

# Delete the folder and everything inside it if it exists
if os.path.exists(base_path):
    shutil.rmtree(base_path)
    
for folder in ["heatmaps", "outlines"]:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

# Initialize storage
y_true = []
y_pred = []
X_test = []
images_lookup = {} # This dictionary stores for each image path, the path to its heatmap and outlined image
counter = 0

for batch in predictions:
    # Basic classification data
    # Note: Using .cpu().numpy() converts the batch tensor to an iterable array
    y_true.extend(batch.gt_label.cpu().numpy())
    y_pred.extend(batch.pred_label.cpu().numpy())
    X_test.extend(batch.image_path)

    # Create the map of different paths for a specific image.
    batch_size = len(batch.image_path)
    for i in range(batch_size):
        img_path = batch.image_path[i]
        image_size = batch.image.shape[-2:]
        
        image = np.array(PILImage.open(img_path).resize(image_size))
        anomaly_map = batch.anomaly_map[i].cpu().numpy().squeeze()

        pred_mask = batch.pred_mask[i].cpu().numpy().squeeze()

        outline_path = f"output_results/outlines/outlined_image_{counter}.png"
        heat_map_path = f"output_results/heatmaps/heat_map_{counter}.png"

        utils.makeOutlineImage(outline_path, pred_mask, image)
        utils.makeHeatMap(heat_map_path, anomaly_map, image)

        # Store the paths of the heatmap and outline image.
        images_lookup[img_path] = {
            "heatMap": heat_map_path,
            "outlined": outline_path,
        }
        counter += 1

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(f"Successfully processed {len(X_test)} images.")

Successfully processed 150 images.


In [8]:
from sklearn.metrics import confusion_matrix

# Compute the confusion matrix
# Labels: 0 = Good, 1 = Broken
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

# Map image paths to their respective CM cells
# results_map[(actual, predicted)] = [list of image paths]
results_map = {(0, 0): [], (0, 1): [], (1, 0): [], (1, 1): []}

for i in range(len(y_true)):
    img_path = X_test[i]
    results_map[(y_true[i], y_pred[i])].append(img_path)

print(results_map)

class_names = ["Good", "Broken"]

{(0, 0): ['C:\\Users\\ierem_wyrt65d\\Documents\\Samuel\\Study\\VUB\\Bachelor_3\\BA3_bachelorproef\\bachelor_thesis\\datasets\\mvtec_anomaly_detection\\cable/test/good/000.png', 'C:\\Users\\ierem_wyrt65d\\Documents\\Samuel\\Study\\VUB\\Bachelor_3\\BA3_bachelorproef\\bachelor_thesis\\datasets\\mvtec_anomaly_detection\\cable/test/good/001.png', 'C:\\Users\\ierem_wyrt65d\\Documents\\Samuel\\Study\\VUB\\Bachelor_3\\BA3_bachelorproef\\bachelor_thesis\\datasets\\mvtec_anomaly_detection\\cable/test/good/002.png', 'C:\\Users\\ierem_wyrt65d\\Documents\\Samuel\\Study\\VUB\\Bachelor_3\\BA3_bachelorproef\\bachelor_thesis\\datasets\\mvtec_anomaly_detection\\cable/test/good/003.png', 'C:\\Users\\ierem_wyrt65d\\Documents\\Samuel\\Study\\VUB\\Bachelor_3\\BA3_bachelorproef\\bachelor_thesis\\datasets\\mvtec_anomaly_detection\\cable/test/good/004.png', 'C:\\Users\\ierem_wyrt65d\\Documents\\Samuel\\Study\\VUB\\Bachelor_3\\BA3_bachelorproef\\bachelor_thesis\\datasets\\mvtec_anomaly_detection\\cable/test/goo

In [9]:
import ipywidgets as widgets
from IPython.display import display

# Load the css file for styling the app
try:
    with open("style.css", "r") as f:
        # Wrap the file content in <style> tags
        css_content = f"""
        <style>
            @import url('https://fonts.googleapis.com/css2?family=Inter:ital,opsz,wght@0,14..32,100..900;1,14..32,100..900&family=Roboto:ital,wght@0,100..900;1,100..900&display=swap');
            {f.read()}
        </style>
        """
    
    # Create the "hidden" widget to inject the styles
    css_widget = widgets.HTML(
        value=css_content, 
        layout=widgets.Layout(display='none')
    )

    # Render it
    display(css_widget)
    print("CSS loaded successfully.")

except FileNotFoundError:
    print("Error: style.css not found. Please check the file path.")

HTML(value="\n        <style>\n            @import url('https://fonts.googleapis.com/css2?family=Inter:ital,op…

CSS loaded successfully.


In [10]:
from IPython.display import display, clear_output
import components as cmp

clear_output(wait=True)

app = cmp.create_ui_interface(
    results_map=results_map, 
    images_lookup=images_lookup, 
    class_names=class_names, 
    cm=cm
)

display(app)